# Titanic - Machine Learning from Disaster
## Baseline Model & Kaggle Submission Pipeline

This notebook demonstrates an end-to-end Machine Learning pipeline for the classic Kaggle Titanic competition:
1. **Exploratory Data Analysis (EDA)**
2. **Feature Engineering** (Title extraction, Family Size, Alone flag)
3. **Missing Value Imputation**
4. **Categorical Encoding**
5. **Model Training & Cross-Validation** (Random Forest Classifier)
6. **Generating Final Submission (`submission.csv`)**

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score

# Set display options
pd.set_option('display.max_columns', None)

## 1. Load Data

In [ ]:
data_dir = os.path.join(".", "data")
train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))
test_df = pd.read_csv(os.path.join(data_dir, "test.csv"))

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
train_df.head()

## 2. Feature Engineering & Preprocessing

We extract meaningful signals:
- **Title** from the `Name` field (e.g., Mr, Mrs, Miss, Master, Rare).
- **FamilySize** = `SibSp` + `Parch` + 1.
- **IsAlone** indicator boolean.
- Impute missing **Age** using median by Pclass and Title.
- Impute missing **Fare** using median by Pclass.
- Impute missing **Embarked** using mode ('S').

In [ ]:
def extract_features(df):
    df = df.copy()
    # Title extraction
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss')
    df['Title'] = df['Title'].replace('Ms', 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')
    
    # Family features
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['Embarked'] = df['Embarked'].fillna('S')
    return df

train_feat = extract_features(train_df)
test_feat = extract_features(test_df)

# Impute Age
age_medians = train_feat.groupby(['Pclass', 'Title'])['Age'].median()
overall_age_median = train_feat['Age'].median()

def fill_age(row):
    if pd.isna(row['Age']):
        val = age_medians.get((row['Pclass'], row['Title']), overall_age_median)
        return overall_age_median if pd.isna(val) else val
    return row['Age']

train_feat['Age'] = train_feat.apply(fill_age, axis=1)
test_feat['Age'] = test_feat.apply(fill_age, axis=1)

# Impute Fare in test
fare_medians = train_feat.groupby('Pclass')['Fare'].median()
overall_fare = train_feat['Fare'].median()
test_feat['Fare'] = test_feat.apply(
    lambda r: fare_medians.get(r['Pclass'], overall_fare) if pd.isna(r['Fare']) else r['Fare'],
    axis=1
)

print("Feature extraction complete.")

## 3. Categorical One-Hot Encoding

In [ ]:
cat_cols = ['Sex', 'Embarked', 'Title']
num_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']

combined = pd.concat([train_feat[cat_cols + num_cols], test_feat[cat_cols + num_cols]], axis=0)
combined_encoded = pd.get_dummies(combined, columns=cat_cols, drop_first=True, dtype=int)

X_train = combined_encoded.iloc[:len(train_feat)].copy()
y_train = train_feat['Survived'].astype(int)
X_test = combined_encoded.iloc[len(train_feat):].copy()

print(f"Final train features shape: {X_train.shape}")
print(f"Final test features shape: {X_test.shape}")
X_train.head()

## 4. Model Training & 5-Fold Stratified Cross-Validation

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=cv, scoring='accuracy')

print(f"5-Fold CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
for fold, score in enumerate(cv_scores, 1):
    print(f"  Fold {fold}: {score:.4f}")

# Fit model on full training set
rf_model.fit(X_train, y_train)

## 5. Feature Importance Analysis

In [ ]:
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top Features:")
print(importance_df)

## 6. Generate Kaggle Submission

In [ ]:
test_predictions = rf_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': test_predictions
})

sub_path = os.path.join(".", "submission.csv")
submission.to_csv(sub_path, index=False)

print(f"Saved submission to: {sub_path}")
print(f"Rows: {len(submission)}")
print(submission.head(10))
print("\nPrediction Distribution:")
print(submission['Survived'].value_counts(normalize=True))